In [1]:
import torch
from transformers import CLIPImageProcessor, CLIPVisionModel
import sys
sys.path.append("../face_anon_simple")

from diffusers import AutoencoderKL, DDPMScheduler
from diffusers.utils import load_image, make_image_grid
from src.diffusers.models.referencenet.referencenet_unet_2d_condition import (
    ReferenceNetModel,
)
from src.diffusers.models.referencenet.unet_2d_condition import UNet2DConditionModel
from src.diffusers.pipelines.referencenet.pipeline_referencenet_final import (
    StableDiffusionReferenceNetPipeline,
)

In [2]:
custom_cache_dir = "face_anon_simple/new_models"  # Change this to a directory with more space

face_model_id = "hkung/face-anon-simple"
clip_model_id = "openai/clip-vit-large-patch14"
sd_model_id = "stabilityai/stable-diffusion-2-1"

print("Start unet download")
unet = UNet2DConditionModel.from_pretrained(
    face_model_id, subfolder="unet", use_safetensors=True, cache_dir=custom_cache_dir
)
print("Finished unet download")

print("Start Reference net download")
referencenet = ReferenceNetModel.from_pretrained(
    face_model_id, subfolder="referencenet", use_safetensors=True, cache_dir=custom_cache_dir
)
print("Finished reference net download")

print("Start condition reference net download")
conditioning_referencenet = ReferenceNetModel.from_pretrained(
    face_model_id, subfolder="conditioning_referencenet", use_safetensors=True, cache_dir=custom_cache_dir
)
print("Finished condition reference net download")

vae = AutoencoderKL.from_pretrained(
    sd_model_id, subfolder="vae", use_safetensors=True, cache_dir=custom_cache_dir
)
scheduler = DDPMScheduler.from_pretrained(
    sd_model_id, subfolder="scheduler", use_safetensors=True, cache_dir=custom_cache_dir
)
feature_extractor = CLIPImageProcessor.from_pretrained(
    clip_model_id, use_safetensors=True, cache_dir=custom_cache_dir
)
image_encoder = CLIPVisionModel.from_pretrained(
    clip_model_id, use_safetensors=True, cache_dir=custom_cache_dir
)

pipe = StableDiffusionReferenceNetPipeline(
    unet=unet,
    referencenet=referencenet,
    conditioning_referencenet=conditioning_referencenet,
    vae=vae,
    feature_extractor=feature_extractor,
    image_encoder=image_encoder,
    scheduler=scheduler,
)

# pipe = pipe.to("cuda")

generator = torch.manual_seed(1)

Start unet download


/projectnb/cs585bp/projects/face_anonymization_proj/.conda/face-anon-simple/lib/python3.8/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Finished unet download
Start Reference net download
Finished reference net download
Start condition reference net download
Finished condition reference net download


In [3]:
from transformers import AutoImageProcessor, ViTForImageClassification
from PIL import Image
import torch
# Load the model and processor
custom_cache_dir_class="../classifier_gender/classifier_model"
model_name = "rizvandwiki/gender-classification-2"

classifier_model = ViTForImageClassification.from_pretrained(model_name, cache_dir=custom_cache_dir_class)
processor = AutoImageProcessor.from_pretrained(model_name, cache_dir=custom_cache_dir_class)

# 'gender_dataset/CelebA_HQ_face_gender_dataset/train/male'
# image_path ="./my_dataset/train/celeb/real/01758_09704.png"
#[Failed for guy3]
# image = Image.open(image_path).convert("RGB") ""

In [4]:
# import face_alignment
# from utils.anonymize_faces_in_image import anonymize_faces_in_image

# # get an input image for anonymization
# original_image = load_image("/projectnb/cs585bp/projects/face_anonymization_proj/face_anon_simple/gender_dataset/CelebA_HQ_face_gender_dataset/train/male/29.jpg")

# # SFD (likely best results, but slower)
# fa = face_alignment.FaceAlignment(
#     face_alignment.LandmarksType.TWO_D, face_detector="sfd"
# )

# # generate an image that anonymizes faces
# anon_image = anonymize_faces_in_image(
#     image=original_image,
#     face_alignment=fa,
#     pipe=pipe,
#     generator=generator,
#     face_image_size=512,
#     num_inference_steps=25,
#     guidance_scale=4.0,
#     anonymization_degree=1.25,
# )
# make_image_grid([original_image, anon_image], rows=1, cols=2)

# Finetuning

In [5]:
import os
import gc
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import save_image
from PIL import Image
import gc 
from tqdm import tqdm, trange
import face_alignment
from utils.anonymize_faces_in_image import anonymize_faces_in_image
from torch.utils.data import Subset
from torch.utils.data import random_split
import numpy as np
import torch.nn.functional as F
import torchvision.transforms.functional as TF
# !pip install matplotlib
import matplotlib.pyplot as plt
from diffusers.utils import load_image


## GPU trial

In [6]:

# Reduce fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:32"
torch.cuda.empty_cache()
fa = face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D, face_detector="sfd")
# Devices
# classifier_device = torch.device("cuda:0")
classifier_device = torch.device("cuda:0")
pipeline_device = torch.device("cuda:0")

# Dataset
transform_raw = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])
dataset = ImageFolder(
    root="/projectnb/cs585bp/projects/face_anonymization_proj/face_anon_simple/gender_dataset/CelebA_HQ_face_gender_dataset/train",
    transform=transform_raw
)
dataset = Subset(dataset, range(100))
train_len = int(0.8 * len(dataset))
val_len = int(0.1 * len(dataset))
test_len = len(dataset) - train_len - val_len
split_generator = torch.Generator().manual_seed(2)

train_set, val_set, test_set = random_split(
    dataset, [train_len, val_len, test_len], generator=split_generator
)


batch_size = 1
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=10, shuffle=False)


# Classifier
classifier_model.to(classifier_device).eval()
for param in classifier_model.parameters():
    param.requires_grad = False


In [7]:
# Pipeline setup
pipe.to(pipeline_device)
pipe.unet.train()
pipe.referencenet.eval()
pipe.conditioning_referencenet.eval()
pipe.vae.eval()
for name, param in pipe.referencenet.named_parameters():
    param.requires_grad = False

for param in pipe.conditioning_referencenet.parameters(): 
    # print("Condition net frozen")
    param.requires_grad = False
for param in pipe.vae.parameters(): 
    # print("Classifier frozen")
    param.requires_grad = False
for n, p in pipe.unet.named_parameters():
    p.requires_grad = False

# Unfreeze last two upsampling blocks
for name, param in pipe.unet.named_parameters():
    if name.startswith("up_blocks.2") or name.startswith("up_blocks.3"):
        param.requires_grad = True

# Unfreeze post-processing conv layers
for name, param in pipe.unet.named_parameters():
    if name in {"conv_norm_out.weight", "conv_norm_out.bias",
                "conv_out.weight", "conv_out.bias"}:
        param.requires_grad = True
    



# Optimizer and loss
# optimizer = optim.Adam(list(pipe.unet.parameters()), lr=1e-4)
optimizer = optim.Adam(list(pipe.referencenet.parameters()), lr=1e-5)
# + list(pipe.conditioning_referencenet.parameters())+ list(pipe.referencene?t.parameters()), lr=1e-4)
criterion = nn.CrossEntropyLoss()
reconstruction_loss_fn = nn.MSELoss()


for name, param in pipe.unet.named_parameters():
    if  param.requires_grad == True:  # <-- modify this
        print(f"✅ Training: {name}")
       
    else:
        # print(f"🧊 Frozen: {name}")
        param.requires_grad = False

# Classifier input transform
transform_for_classifier = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])


✅ Training: up_blocks.2.attentions.0.norm.weight
✅ Training: up_blocks.2.attentions.0.norm.bias
✅ Training: up_blocks.2.attentions.0.proj_in.weight
✅ Training: up_blocks.2.attentions.0.proj_in.bias
✅ Training: up_blocks.2.attentions.0.transformer_blocks.0.norm1.weight
✅ Training: up_blocks.2.attentions.0.transformer_blocks.0.norm1.bias
✅ Training: up_blocks.2.attentions.0.transformer_blocks.0.attn1.to_q.weight
✅ Training: up_blocks.2.attentions.0.transformer_blocks.0.attn1.to_k.weight
✅ Training: up_blocks.2.attentions.0.transformer_blocks.0.attn1.to_v.weight
✅ Training: up_blocks.2.attentions.0.transformer_blocks.0.attn1.to_out.0.weight
✅ Training: up_blocks.2.attentions.0.transformer_blocks.0.attn1.to_out.0.bias
✅ Training: up_blocks.2.attentions.0.transformer_blocks.0.norm2.weight
✅ Training: up_blocks.2.attentions.0.transformer_blocks.0.norm2.bias
✅ Training: up_blocks.2.attentions.0.transformer_blocks.0.attn2.to_q.weight
✅ Training: up_blocks.2.attentions.0.transformer_blocks.0.at

In [8]:


# Training loop
num_epochs = 3
generator = torch.manual_seed(1)
torch.manual_seed(1)
np.random.seed(1)

train_losses = []
val_losses = []


for epoch in trange(num_epochs, desc="Training Epochs"):
    print(f"\n🌀 Epoch {epoch+1}/{num_epochs}")
    epoch_train_loss = 0.0
    total_train_batches = 0
    epoch_val_loss = 0.0
    total_val_batches = 0


    for i, (image_tensor, label) in enumerate(train_loader):
        try:
            gc.collect()
            torch.cuda.empty_cache()
        
            ce_loss = 0
            mse_losses = []
            j = 0
            # for j in range(image_tensor.shape[0]):
            original_image = transforms.ToPILImage()(image_tensor[j]).convert("RGB")

            print(torch.cuda.memory_allocated(0) / 1024**2, "MB on GPU 1")
            # print(torch.cuda.memory_allocated(1) / 1024**2, "MB on GPU 1")

            generated_image_tensor  = pipe(
                source_image=original_image,
                conditioning_image=original_image,
                num_inference_steps=5,
                guidance_scale=4.0,
                generator=generator,
                output_type='pt',
                anonymization_degree=1.25,
            ).images[0]   
            print(torch.cuda.memory_allocated(0) / 1024**2, "MB on GPU 1")
            # print(torch.cuda.memory_allocated(1) / 1024**2, "MB on GPU 1")

            # generated_image_tensor = transforms.ToTensor()(output).to(classifier_device)
            # composite = composite.to(classifier_device, non_blocking=True)

            processed_tensor = transform_for_classifier(generated_image_tensor)
            inputs = {"pixel_values": processed_tensor.unsqueeze(0)}
            logits = classifier_model(**inputs).logits
            target = label[j].unsqueeze(0).to(classifier_device)
            original_resized = transforms.Resize(generated_image_tensor.shape[-2:])(image_tensor[j].squeeze(0)).to(classifier_device)
            ce_loss += criterion(logits, target)

            
            print("Saving")
            out_dir = f"outputs_naman/gender_finetune/epoch_{epoch}"
            os.makedirs(out_dir, exist_ok=True)
            save_path = os.path.join(out_dir, f"naman_image_{j}_label_{label[j].item()}.png")
            save_path_og = os.path.join(out_dir, f"naman_image_{j}_label_{label[j].item()}_original.png")
            # Make sure to move back to CPU & clamp [0,1]
            save_image(generated_image_tensor.detach().cpu().clamp(0, 1), save_path)
            save_image(image_tensor[j].cpu(), save_path_og)
            ###########################################################

            loss = ce_loss/batch_size
            
            optimizer.zero_grad()
            loss.backward()
            for name, param in pipe.referencenet.named_parameters():
                    if param.requires_grad:
                        print(f"{name}: grad is {'✅ set' if param.grad is not None else '❌ None'}")
                        break
            print("Loss grad_fn:", loss.grad_fn) 
            optimizer.step()
            # for name, param in pipe.unet.named_parameters():
            #     if param.requires_grad:
            #         if not torch.equal(param.detach(), initial_weights[name]):
            #             print(f"✅ {name} was updated.")
            #         else:
            #             print(f"❌ {name} did NOT change.")
            #         break
            # updated_weights = pipe.unet.state_dict()
            epoch_train_loss += loss.item()
            total_train_batches += 1     

            if i % 1 == 0:
                print(f"  Batch {i:03d} | Loss: {loss.item():.4f}")
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(e)
                print(f"⚠️ OOM at batch {i}, skipping")
                torch.cuda.empty_cache()
                raise e 
                continue
            else:
                print("ERRORRRRRR : ",e)
                # continue
                raise e
    
    save_dir = f"ft_checkpoints/epoch_{epoch}"
    os.makedirs(save_dir, exist_ok=True)
    torch.save(pipe.unet.state_dict(), os.path.join(save_dir, "unet.pt"))
    print(f"💾 Saved UNet at {save_dir}")
    np.save(f"train_losses_epoch{epoch}.npy", np.array(train_losses))
    np.save(f"val_losses_epoch{epoch}.npy", np.array(val_losses))

Training Epochs:   0%|          | 0/3 [00:00<?, ?it/s]


🌀 Epoch 1/3
12109.55419921875 MB on GPU 1
42776.78515625 MB on GPU 1
Saving


/projectnb/cs585bp/projects/face_anonymization_proj/.conda/face-anon-simple/lib/python3.8/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


Loss grad_fn: <DivBackward0 object at 0x152f0b229fa0>
  Batch 000 | Loss: 0.0041
13676.72314453125 MB on GPU 1


Training Epochs:   0%|          | 0/3 [00:14<?, ?it/s]

CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacty of 44.43 GiB of which 112.62 MiB is free. Including non-PyTorch memory, this process has 44.31 GiB memory in use. Of the allocated memory 43.84 GiB is allocated by PyTorch, and 124.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF
⚠️ OOM at batch 1, skipping


OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacty of 44.43 GiB of which 112.62 MiB is free. Including non-PyTorch memory, this process has 44.31 GiB memory in use. Of the allocated memory 43.84 GiB is allocated by PyTorch, and 124.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF